## **Load Data**
---

In [1]:
# Import libraries
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_parquet("../data/processed/acs_cleaned.parquet")

df = df.rename(columns={
    "Log Income": "Log_Income"
})

df.shape

(15715097, 14)

In [3]:
# Create age squared for lifecycle earnings
df["Age2"] = df["Age"]**2

## **Sampling**
---

In [4]:
df_model = df.sample(500000, random_state = 123)

## **Baseline Model - Education Only**
---

In [5]:
model1 = smf.wls(
    "Log_Income ~ C(Q('Education Group'))",
    data=df_model,
    weights=df_model["Individuals Represented"]
).fit()

print(model1.summary())

                            WLS Regression Results                            
Dep. Variable:             Log_Income   R-squared:                       0.169
Model:                            WLS   Adj. R-squared:                  0.169
Method:                 Least Squares   F-statistic:                 3.397e+04
Date:                Wed, 18 Mar 2026   Prob (F-statistic):               0.00
Time:                        10:21:57   Log-Likelihood:                   -inf
No. Observations:              500000   AIC:                               inf
Df Residuals:                  499996   BIC:                               inf
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                                                      coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------------

/opt/anaconda3/lib/python3.13/site-packages/statsmodels/regression/linear_model.py:806: RuntimeWarning: divide by zero encountered in log
  llf += 0.5 * np.sum(np.log(self.weights))


### **Model**

$$log(Income) = \beta_0 + \beta_1EducationGroup$$

The dependent variable is log real income, adn the model estimates the relationship between education level and wages using weighted least squares with ACS sampling weights.

The regression is estimated on 500,000 observations sampled from the cleaned dataset

### **Reference Group**

The omitted category is Bachelor's degree holders, meaning all coefficients represent differences relative to Bachelor's degrees.

### **Interpretation of Coefficients**

#### HS or Less
Coefficient:
$$-0.6988$$

Exponentiating to convert from log differences:
$$e^{-0.6988}=0.50$$

Interpretation:
Individuals with a high school diploma or less earn roughly 50% lower wages than Bachelor's degree holders on average.

#### Some College
Coefficient:
$$-0.4177$$

Exponentiating to convert from log differences:
$$e^{-0.4177}=0.66$$

Interpretation:
Individuals with some college education earn rabout 34% less than those with a Bachelor's degree.

#### Masters or Doctorate
Coefficient:
$$0.2538$$

Exponentiating to convert from log differences:
$$e^{0.2538}=1.29$$

Interpretation:
Individuals with graduate degrees earn approximately 29% more than Bachelor's degree holders.

#### Intercept
Coefficient:
$$11.2989$$

Exponentiating to convert from log differences:
$$e^{11.2989}=80,800$$

Interpretation:
The predicted average annual income for Bachelor's degree holders is roughly $81,000 in 2024 dollars.

### **Statistical Significance**

All education coefficients are:
$$p<0.001$$

This means the estimated wage differences across education groups are highly statistically significant.

### **Model Fit**

$$R^2=0.169$$

Interpretation: Education alone explains about 17% of the variation in wages in the dataset.

This is expected because wages are also influenced by factors such as:
* Experience
* Occupation
* Geographic location
* Industry
* Demographic characteristics

## **Add Experience (Age)**
---

In [6]:
model2 = smf.wls(
    "Log_Income ~ C(Q('Education Group')) + Age + Age2",
    data=df_model,
    weights=df_model["Individuals Represented"]
).fit()

print(model2.summary())

                            WLS Regression Results                            
Dep. Variable:             Log_Income   R-squared:                       0.189
Model:                            WLS   Adj. R-squared:                  0.189
Method:                 Least Squares   F-statistic:                 2.336e+04
Date:                Wed, 18 Mar 2026   Prob (F-statistic):               0.00
Time:                        10:21:57   Log-Likelihood:                   -inf
No. Observations:              500000   AIC:                               inf
Df Residuals:                  499994   BIC:                               inf
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                                                      coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------------

/opt/anaconda3/lib/python3.13/site-packages/statsmodels/regression/linear_model.py:806: RuntimeWarning: divide by zero encountered in log
  llf += 0.5 * np.sum(np.log(self.weights))


### **Interpretation**

This model expands upon the baseline regression by adding age and age squared to approximate work experience.

The results show that education remains a strong predictor of wages. Relative to individuals with a Bachelor's degree:
* Individuals with HS or less earn substantially lower wages.
* Individuals with some college also learn less than Bachelor's degree holders.
* Individuals with Master's or Doctorate degrees earn higher wages than those with Bachelor's degrees.

Adding age variables improves the model fit with $R^2$ increasing from 0.169 to 0.189, indicating that experience explains additional variation in income.

The age coefficient is positive, meaning wages increase with experience. The age squared coefficient captures the curvature of the wage-experience relationship, implying that wage growth slows later in life.

## **Add Demographics**
---

In [7]:
model3 = smf.wls(
    """Log_Income ~ 
    C(Q('Education Group')) +
    Age + Age2 +
    C(Sex) +
    C(Demographic)
    """,
    data=df_model,
    weights=df_model["Individuals Represented"]
).fit()

print(model3.summary())

                            WLS Regression Results                            
Dep. Variable:             Log_Income   R-squared:                       0.246
Model:                            WLS   Adj. R-squared:                  0.246
Method:                 Least Squares   F-statistic:                 1.631e+04
Date:                Wed, 18 Mar 2026   Prob (F-statistic):               0.00
Time:                        10:21:58   Log-Likelihood:                   -inf
No. Observations:              500000   AIC:                               inf
Df Residuals:                  499989   BIC:                               inf
Df Model:                          10                                         
Covariance Type:            nonrobust                                         
                                                      coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------------

/opt/anaconda3/lib/python3.13/site-packages/statsmodels/regression/linear_model.py:806: RuntimeWarning: divide by zero encountered in log
  llf += 0.5 * np.sum(np.log(self.weights))


### **Interpretation**

This model adds gender and demographic controls to the wage regression alongside education and experience. The model fit improves further, with $R^2$ increasing to 0.246, indicating that demographic characteristics explain additional variation in wages.

Education continues to have a strong effect on earnings. Compared to individuals with a Bachelor’s degree, those with HS or less or some college earn significantly lower wages, while those with Master’s or Doctorate degrees earn higher wages.

The results also reveal a gender wage gap: holding education and experience constant, men earn significantly more than women on average.

Demographic differences are also present. Relative to the omitted demographic category, Black and Hispanic individuals earn lower wages on average, even after controlling for education and experience. 

The positive coefficient on age combined with the small age² term again reflects the typical concave age–earnings profile, where wages increase with experience but grow more slowly later in life.

## **Add Geographic Controls**
---

In [8]:
model4 = smf.wls(
    """Log_Income ~ 
    C(Q('Education Group')) +
    Age + Age2 +
    C(Sex) +
    C(Demographic) +
    C(State)
    """,
    data=df_model,
    weights=df_model["Individuals Represented"]
).fit()

print(model4.summary())

                            WLS Regression Results                            
Dep. Variable:             Log_Income   R-squared:                       0.262
Model:                            WLS   Adj. R-squared:                  0.262
Method:                 Least Squares   F-statistic:                     2963.
Date:                Wed, 18 Mar 2026   Prob (F-statistic):               0.00
Time:                        10:22:01   Log-Likelihood:                   -inf
No. Observations:              500000   AIC:                               inf
Df Residuals:                  499939   BIC:                               inf
Df Model:                          60                                         
Covariance Type:            nonrobust                                         
                                                      coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------------

/opt/anaconda3/lib/python3.13/site-packages/statsmodels/regression/linear_model.py:806: RuntimeWarning: divide by zero encountered in log
  llf += 0.5 * np.sum(np.log(self.weights))


### **Interpretation**

This model adds state fixed effects to control for geographic differences in labor markets, such as cost of living, regional economic conditions, and industry composition. Including these controls increases the model fit further, with $R^2$ rising to 0.262, indicating that geographic variation explains additional differences in wages.

Education continues to be a major determinant of earnings. Relative to individuals with a Bachelor’s degree, those with HS or less and some college earn significantly lower wages, while individuals with Master’s or Doctorate degrees earn higher wages.

The gender wage gap persists, with men earning higher wages than women even after controlling for education, experience, demographics, and location.

Demographic wage disparities also remain: Black and Hispanic workers earn lower wages on average, suggesting persistent wage differences that cannot be fully explained by education, experience, or geographic location.

State fixed effects reveal substantial regional variation in wages. High-income states such as California, New York, Massachusetts, and New Jersey show significantly higher wage levels relative to the baseline state, while several lower-income states have negative or smaller wage effects.

The age and age² coefficients again produce the expected concave age–earnings profile, where wages increase with experience but grow more slowly later in life.

## **Add Time Controls**
---

In [9]:
model5 = smf.wls(
    """Log_Income ~ 
    C(Q('Education Group')) +
    Age + Age2 +
    C(Sex) +
    C(Demographic) +
    C(State) +
    C(Year)
    """,
    data=df_model,
    weights=df_model["Individuals Represented"]
).fit()

print(model5.summary())

                            WLS Regression Results                            
Dep. Variable:             Log_Income   R-squared:                       0.263
Model:                            WLS   Adj. R-squared:                  0.263
Method:                 Least Squares   F-statistic:                     2378.
Date:                Wed, 18 Mar 2026   Prob (F-statistic):               0.00
Time:                        10:22:04   Log-Likelihood:                   -inf
No. Observations:              500000   AIC:                               inf
Df Residuals:                  499924   BIC:                               inf
Df Model:                          75                                         
Covariance Type:            nonrobust                                         
                                                      coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------------

/opt/anaconda3/lib/python3.13/site-packages/statsmodels/regression/linear_model.py:806: RuntimeWarning: divide by zero encountered in log
  llf += 0.5 * np.sum(np.log(self.weights))


### **Interpretation**

This final specification includes state and year fixed effects, allowing the model to control for both geographic differences and economy-wide changes over time. With these additional controls, the model fit improves slightly to $R^2$ = 0.263, meaning the included variables explain about 26% of the variation in wages.

Education remains a strong predictor of income. Relative to individuals with a Bachelor’s degree, those with HS or less or some college earn substantially lower wages, while individuals with Master’s or Doctorate degrees earn higher wages.

The gender wage gap persists, with men earning significantly higher wages than women even after controlling for education, experience, demographics, location, and time.

Demographic wage disparities also remain. Black and Hispanic individuals earn lower wages on average, indicating that these wage differences cannot be fully explained by observable characteristics such as education, experience, or geography.

State fixed effects reveal substantial regional variation in wages, reflecting differences in labor markets and cost of living across states. High-income states and metropolitan regions tend to show higher wage levels relative to the baseline state.

Year fixed effects capture macroeconomic changes over time.

Finally, the age and age² coefficients continue to produce the expected concave age–earnings profile, where wages increase with experience but the rate of growth slows later in life.

## **Major Premium**
---

In [10]:
major_df = df_model.dropna(subset=["Major"])

major_model = smf.wls(
    "Log_Income ~ C(Major) + Age + Age2 + C(Sex)",
    data=major_df,
    weights=major_df["Individuals Represented"]
).fit()

print(major_model.summary())

                            WLS Regression Results                            
Dep. Variable:             Log_Income   R-squared:                       0.119
Model:                            WLS   Adj. R-squared:                  0.118
Method:                 Least Squares   F-statistic:                     282.5
Date:                Wed, 18 Mar 2026   Prob (F-statistic):               0.00
Time:                        10:22:05   Log-Likelihood:                -94136.
No. Observations:               81721   AIC:                         1.884e+05
Df Residuals:                   81681   BIC:                         1.887e+05
Df Model:                          39                                         
Covariance Type:            nonrobust                                         
                                                        coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------

### **Interpretation**

This model examines how college major influences earnings, controlling for age, experience (age squared), and gender. The regression uses individuals with reported majors and estimates the relationship between field of study and log income.

The results show substantial variation in wages across different college majors. Several technical and quantitative fields have the highest positive wage premiums. In particular, Engineering, Computer and Information Sciences, Mathematics and Statistics, and Physical Sciences show large positive coefficients, indicating significantly higher predicted earnings relative to the baseline major.

Business, Biology/Life Sciences, and History also show positive wage effects, though generally smaller than those observed for engineering and quantitative fields.

In contrast, some majors are associated with lower predicted wages, including Culinary and Cosmetology, Theology, and Military Technologies. These negative coefficients suggest lower average earnings relative to the baseline field of study.

The model also confirms a gender wage gap, with men earning significantly more than women after controlling for major and experience.